# 01. Your first experiment

This notebook walks through a complete experiment with `skxperiments`: **design** the randomization, **estimate** the effect, and **infer** whether it is real.

> Library philosophy: *the treatment assignment mechanism is the starting point*. We first decide **how** to randomize; the statistical model comes second.

## 1. Potential outcomes

The conceptual basis is the *Rubin Causal Model*: each unit has two potential outcomes, `Y(0)` (without treatment) and `Y(1)` (with treatment). We only observe **one** of them per unit; the other is counterfactual.

For this example we simulate both, with a constant true effect of `+0.5`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 200

y0 = rng.normal(size=n)   # Y(0): outcome without treatment
tau = 0.5                 # true effect (constant)
y1 = y0 + tau             # Y(1): outcome with treatment

df = pd.DataFrame({"x": rng.normal(size=n)})   # one pre-experiment covariate
df.head()

## 2. Design: completely randomized design (CRD)

`CRD` assigns half the units to treatment, at random. Randomization is what lets us attribute outcome differences to the treatment rather than to confounders.

In [ ]:
from skxperiments.design.crd import CRD

design = CRD(p=0.5, seed=42)
assignment = design.randomize(df)
assignment.n_treated_, assignment.n_control_

## 3. What we observe

We reveal `Y(1)` for the treated and `Y(0)` for the controls. That is the only outcome that exists in practice. The outcome **belongs to the data**, so we build a new `Assignment` with the `y` column (without mutating the original).

In [ ]:
from skxperiments.core.assignment import CRDAssignment

t = assignment.data_[assignment.treatment_col_].to_numpy()
observed = assignment.data_.copy()
observed["y"] = np.where(t == 1, y1, y0)

assignment = CRDAssignment(
    data=observed,
    treatment_col=assignment.treatment_col_,
    design=design,
    seed=42,
)

## 4. Estimate the ATE

`DifferenceInMeans` is the simplest estimator: mean of the treated minus mean of the controls. It should land near the true effect of `0.5`.

In [ ]:
from skxperiments.estimators.difference_in_means import DifferenceInMeans

result = DifferenceInMeans(outcome_col="y").fit(assignment).estimate()
print(f"Estimated ATE: {result.ate:.3f}  (true: {tau})")

## 5. Randomization-based inference

`RandomizationTest` tests Fisher's *sharp* null (no effect for any unit) by repeating the randomization thousands of times. The p-value does not rely on distributional assumptions; it comes only from the randomization mechanism **we** chose.

In [ ]:
from skxperiments.inference import RandomizationTest

rt = RandomizationTest(
    estimator=DifferenceInMeans(outcome_col="y"),
    n_permutations=500,
    seed=0,
)
res = rt.fit(assignment).estimate()
print(f"ATE: {res.ate:.3f}  |  p-value: {res.p_value:.4f}")

## 6. Visualize the null distribution

The observed ATE (red line) should fall in the tail of the distribution of ATEs under the null. That is what makes the effect "significant".

In [ ]:
from skxperiments.reporting import plot_null_distribution

ax = plot_null_distribution(res)
ax.figure

## What we learned

- The causal effect is defined by **potential outcomes**; randomization reveals one per unit.
- `CRD`, `DifferenceInMeans`, and `RandomizationTest` form the minimal end-to-end flow.
- The randomization p-value assumes no normality; it comes from the design itself.

**Next:** `02. Inference three ways` compares randomization, Neyman (finite population), and bootstrap (superpopulation): when to use each.